<a href="https://colab.research.google.com/github/takeshun1984/kyoshin-koushu/blob/main/OpenSWPCtest.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

OpenSWPCをGoogle Colab上で、makeして動かしてみることで、環境構築、ビルド、並列計算を体験していただきます。

OpenSWPC: https://github.com/OpenSWPC/OpenSWPC  
オンラインマニュアル： https://openswpc.github.io/legacy/ja/  
論文： https://doi.org/10.1186/s40623-017-0687-2  
書籍： https://www.utp.or.jp/book/b10171082.html

動画をcolab上で見るための下準備として、以下を実施します。

In [ ]:
from IPython.display import HTML
from base64 import b64encode

def show_mp4(mp4file):
  mp4 = open(mp4file, 'rb').read()
  data_url = 'data:video/mp4;base64,' + b64encode(mp4).decode()

  html=HTML(f"""
<video width="50%" height="50%" controls>
   <source src="{data_url}" type="video/mp4">
</video>""")

  return html

OpenSWPCの（Colab上への）インストールに必要なライブラリをインストールします。Google Colab上では、Pythonコードはそのまま記述することができますが、Linuxの命令などは「!」や「%」をつけることで、利用できます。

例）  
ファイルの検索
```
! ls
```
（状態が保持される）ディレクトリの移動
```
%cd
```
＊「!cd」では次の行では状態が保持されない。






In [ ]:
! apt install libnetcdf-dev libnetcdff-dev

GitHubからOpenSWPCの最新版をダウンロード

In [ ]:
! git clone https://github.com/OpenSWPC/OpenSWPC

Google ColabはUbuntuとなります。
```
! cat /etc/os-release
```
で、確認することができます。そのため、[makefileの指定変数](https://openswpc.github.io/legacy/ja/1._SetUp/0103_compile/#makefile)より、ubuntu-gfortranを選択してmakeコマンドを実行します。



In [ ]:
! cd ./OpenSWPC/src; make arch=ubuntu-gfortran

makeの結果を確認するために、実行バイナリができているかを確認します。

In [ ]:
! ls OpenSWPC/bin/

サンプルのインプットファイルで2次元計算を実行するために、「OpenSWPC」へディレクトリを移動します。

In [ ]:
%cd OpenSWPC/

ターミナルを用いて「example/input.inf」を開くか、  
https://github.com/OpenSWPC/OpenSWPC/blob/master/example/input.inf  
からインプットファイルのサンプルを確認できます。

OpenSWPCでは2次元計算であっても、3次元計算であっても同様のinput fileで動作します。実行時に「swpc_3d」「swpc_psv」「swpc_sh」を選択するだけで、3次元、2次元PSV、2次元SH問題の実行が可能です。2次元計算の場合は、xz平面のみ実行されます。以下の例では、xyzの座標系で水平成層構造（F-net 1D; Kubo et al. 2002）の計算を実施しますが、緯度経度深さ系での3次元構造（境界面深度が水平方向に変化する層構造）の計算では、計算領域の中心を通るxz平面の計算が実行されます。  
詳細は、オンラインマニュアルの「[2次元コード固有の設定](https://openswpc.github.io/legacy/ja/2._Parameters/0211_2dcode/)」をご覧ください。

In [ ]:
! mpirun --allow-run-as-root --oversubscribe -np 2 ./bin/swpc_psv.x -i example/input.inf

出力された波動場（PSV）を以下の命令で、ppm形式の画像データへ変換します。

In [ ]:
! ./bin/read_snp.x -i out/swpc.psv.xz.ps.nc -ppm -pall -mul 1e5

ffmpegを用いて動画に変更します。

In [ ]:
! ffmpeg -i swpc/psv/xz/ps/swpc.psv.xz.ps.000%3d.ppm -qscale 0 -pix_fmt yuv420p -y swpc.mp4

In [ ]:
show_mp4("./swpc.mp4")

次に、計算された地震波形を確認します。OpenSWPCではSAC形式で出力されます。


```
  !! ----------------------------------------------------------------------- !!
  !! Waveform Output
  !!

  sw_wav_v         = .true.           !! velocity trace output at stations
  sw_wav_u         = .false.          !! displacement trace output at stations
  sw_wav_stress    = .false.           !! stress tensor trace
  sw_wav_strain    = .false.           !! strain tansor trace
  ntdec_w          = 5                !! time decimation of waveform output
  st_format        = 'xy'             !! station format: 'xy' or 'll'
  fn_stloc         = './example/stloc.xy'  !! station location file
  wav_format       = 'sac'            !! 'sac' or 'csf' ('sac' recommended)
  ntdec_w_prg      = 0                !!  waveform output during computation (0:off)
```
とinput.infにはありますので、速度波形が出力されます。obspyをインストールして、SAC形式のファイルを読み込みます。

[stloc.xy](https://github.com/OpenSWPC/OpenSWPC/blob/master/example/stloc.xy)は、


```
#                                                    -*- mode:sh -*-
# stloc.xy
#
# station location data by cartesian format.
# lines starting from '#' and blank lines are omitted.
#
# zsw: controls station depth
#      'dep': use the depth
#      'fsb': locate one-grid below from the free surface/sea surface
#      'obb': locate one-grid below from the groud surface/seafloor
#      'oba': locate one-grid above from the groud surface/seafloor
#      'bd{i}' (i=0,...,9) i-th boundary interface
#
# ---
#      x     y     z       stnm   zsw
# --------------------------------------
     0.0   0.0   0.0       st01   obb
   -10.0  -5.0   0.0       st02   obb
    10.0   5.0   0.0       st03   obb
```
となっていますので、st02とst03を描画して比較します。



In [ ]:
! pip install obspy

In [ ]:
from obspy import read

st = read('out/wav/swpc.psv.st0?.Vz.sac',fromat='SAC')

st.plot()